In [ ]:
# COLAB SETUP

%cd /content
!rm -rf /content/proto-tsrl
!git clone https://github.com/haiyan-wang/proto-tsrl.git /content/proto-tsrl
%cd /content/proto-tsrl

from google.colab import drive
drive.mount('/content/drive')

import sys
import os

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(project_root)

In [ ]:
import os 

import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

from src.utils.sampling_utils import TimeSeriesDataset
from src.experiments.ppg.ppg_model import PPGModel

In [ ]:
# SETTINGS

SEED = 42

# device
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(DEVICE)

# data quality
INCLUDE_CLEAN_DATA = False
INCLUDE_SEMINOISY_DATA = True
INCLUDE_NOISY_DATA = False

# architecture 
REP_DIMS = [10, 50, 100]
MODELS = {}
for dim in REP_DIMS:
    model_list = {}
    
    CKPT_DIR = f"/content/drive/MyDrive/Duke/Senior Year/Thesis/experiments/ppg/dim{dim}"
    for ckpt_file in os.listdir(CKPT_DIR):
        ckpt = torch.load(f"/content/drive/MyDrive/Duke/Senior Year/Thesis/experiments/ppg/dim{dim}/{ckpt_file}", map_location = "cpu")
        MODEL = PPGModel(representation_dimension = dim)
        MODEL.load_state_dict(ckpt)
        
        MODEL = MODEL.to(DEVICE)
        if torch.cuda.device_count() > 1:
            MODEL = nn.DataParallel(MODEL)
        elif not isinstance(MODEL, nn.DataParallel):
            MODEL = nn.DataParallel(MODEL)

        MODEL.eval()

        model_list[ckpt_file] = MODEL
    
    MODELS[dim] = model_list 

In [ ]:
# DATA LOADING FUNCTIONS

# quality: high - [0,0,1] med - [0,1,0] low - [1,0,0]
# afib: pos - [0,1]
def load_data(
        clean : bool,
        seminoisy : bool,
        noisy : bool,
        dataset : str,
        return_labels : bool = True
    ):
    """
    Load PPG data from Drive

    Arguments
    ---------
        - clean, seminoisy, noisy: bools indicating whether to include data of each quality level
        - dataset: 'train' or 'test'
        - return_labels: whether to return labels (if False, only returns signals)

    Returns
    -------
        - X: (N, L) ndarray of PPG signals
        - y: (N,) ndarray of binary labels indicating presence of afib if return_labels = True
    """

    file_path = '/content/drive/MyDrive/Duke/Senior Year/Thesis/ppg_data/'
    with np.load(file_path + f'{dataset}.npz') as data:
        ppg_signal = data['signal']
        ppg_qual = data['qa_label']
        rhythm = data['rhythm']

    qual_mask = np.zeros(ppg_qual.shape[0], dtype = bool)
    if clean:
        qual_mask = (qual_mask | np.all(ppg_qual == np.array([0, 0, 1]), axis = 1))
    if seminoisy:
        qual_mask = (qual_mask | np.all(ppg_qual == np.array([0, 1, 0]), axis = 1))
    if noisy:
        qual_mask = (qual_mask | np.all(ppg_qual == np.array([1, 0, 0]), axis = 1))

    X, y = ppg_signal[qual_mask], rhythm[qual_mask]

    afib_label = np.array([0, 1])
    y = np.all(y == afib_label, axis = 1)

    if return_labels:
        return X, y

    return X

In [ ]:
X_test, y_test = load_data(
    clean = INCLUDE_CLEAN_DATA,
    seminoisy = INCLUDE_SEMINOISY_DATA,
    noisy = INCLUDE_NOISY_DATA,
    dataset = 'train',
    return_labels = True
)

sig_test = [torch.from_numpy(x).float() for x in X_test]
ds_test = TimeSeriesDataset(sig_test)
dl_test = DataLoader(
    ds_test,
    batch_size = X_test.shape[0],
    shuffle = True,
    num_workers = 4,
    pin_memory = True
)

In [ ]:
with torch.inference_mode():
    X_test = next(iter(dl_test))
    X_test = X_test.to(DEVICE, non_blocking = True)

    for dim in REP_DIMS:
        model_dict = MODELS[dim]

        for epoch, model in model_dict.items(): 
            repr = model(X_test)
            break 

print(type(repr))
print(repr.shape)